In [ ]:
import numpy as np
import re

sentence = "OMG!!! AI students rrr brillianttt 😊"

print("Original sentence:")
print(sentence)

text = sentence.lower()
text = re.sub(r"[^a-z\s]", " ", text)
text = text.replace("rrr", "are")
text = text.replace("brillianttt", "brilliant")
text = " ".join(text.split())

print("\nAfter preprocessing:")
print(text)

tokens = text.split()

print("\nTokens:")
print(tokens)

vocab = {
    "[PAD]": 0,
    "[UNK]": 1
}

for word in tokens:
    if word not in vocab:
        vocab[word] = len(vocab)

token_ids = [vocab.get(word, vocab["[UNK]"]) for word in tokens]

print("\nVocabulary:")
print(vocab)

print("\nToken IDs:")
print(token_ids)

embedding_dim = 8
np.random.seed(42)

embedding_matrix = np.random.randn(
    len(vocab),
    embedding_dim
)

X = embedding_matrix[token_ids]

print("\nToken embeddings:")
print(np.round(X, 3))

def positional_encoding(sequence_length, d_model):
    PE = np.zeros((sequence_length, d_model))

    for pos in range(sequence_length):
        for i in range(0, d_model, 2):
            PE[pos, i] = np.sin(
                pos / (10000 ** (i / d_model))
            )

            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(
                    pos / (10000 ** (i / d_model))
                )

    return PE

PE = positional_encoding(len(tokens), embedding_dim)

print("\nPositional encoding:")
print(np.round(PE, 3))

Z = X + PE

print("\nTransformer input (X + PE):")
print(np.round(Z, 3))

d_model = embedding_dim
d_k = 4

W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

Q = Z @ W_Q
K = Z @ W_K
V = Z @ W_V

print("\nQ:")
print(np.round(Q, 3))

print("\nK:")
print(np.round(K, 3))

print("\nV:")
print(np.round(V, 3))

scores = Q @ K.T
scores = scores / np.sqrt(d_k)

print("\nAttention scores:")
print(np.round(scores, 3))

def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

attention_weights = softmax(scores)

print("\nAttention weights:")
print(np.round(attention_weights, 3))

print("\nAttention row sums:")
print(np.sum(attention_weights, axis=1))

attention_output = attention_weights @ V

print("\nAttention output:")
print(np.round(attention_output, 3))

W_O = np.random.randn(d_k, d_model) * 0.1

projected_output = attention_output @ W_O

print("\nOutput projection:")
print(np.round(projected_output, 3))

residual_1 = Z + projected_output

print("\nAfter first residual connection:")
print(np.round(residual_1, 3))

def layer_norm(x):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    epsilon = 1e-6

    return (
        (x - mean)
        / np.sqrt(variance + epsilon)
    )

normalized_1 = layer_norm(residual_1)

print("\nAfter first LayerNorm:")
print(np.round(normalized_1, 3))

hidden_dim = 16

W1 = np.random.randn(d_model, hidden_dim) * 0.1
W2 = np.random.randn(hidden_dim, d_model) * 0.1

hidden = normalized_1 @ W1

def gelu(x):
    return 0.5 * x * (
        1 + np.tanh(
            np.sqrt(2 / np.pi)
            * (x + 0.044715 * x ** 3)
        )
    )

hidden = gelu(hidden)
ffn_output = hidden @ W2

print("\nFFN output:")
print(np.round(ffn_output, 3))

residual_2 = normalized_1 + ffn_output

print("\nAfter second residual connection:")
print(np.round(residual_2, 3))

final_output = layer_norm(residual_2)

print("\nFinal contextual representations:")
print(np.round(final_output, 3))

print("\nWord representations:")

for word, vector in zip(tokens, final_output):
    print(
        f"{word:10s} ->",
        np.round(vector, 3)
    )
